# Customer Churn: Bagging vs Boosting

**Ensemble methods on a real retention problem**

A single tree model is easy to read but fragile: grown deep it **memorises** the training data and
then swings wildly on data it hasn't seen. This notebook fixes that by combining many trees into an
**ensemble**, and contrasts the two great families for doing so **bagging** and **boosting** on
a classic business problem: predicting which telecom customers will **churn** (cancel their
service).

We use the *Telco Customer Churn* dataset (7,043 customers). About **27%** churn, so accuracy alone
is misleading a model that predicts *"nobody leaves"* is already 73% accurate and worthless. As
in earlier weeks we judge everything on **F1 and recall for the "Churn" class**, measured on **both
the training and test sets** so the train/test *gap* the fingerprint of overfitting is always
in view.

**The journey:**

| Step | What we do | Why |
|---|---|---|
| **1. Explore** | Exploratory Data Analysis | Understand the data before modelling |
| **2. Preprocess** | Encode **all** features, stratified split | Do transformations as per observations in EDA |
| **3. Baseline** | A single Decision Tree | See high variance / overfitting first-hand |
| **4. Bagging** | `BaggingClassifier`, `RandomForest` | Reduce **variance** by averaging many trees |
| **5. Boosting** | `AdaBoost`, `XGBoost` | Reduce **bias** by correcting mistakes in sequence |

---

## Setup: Imports



In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

from sklearn.model_selection import train_test_split

## 1: Load & Explore the Data

The loader tries a local file first, then a public mirror, so the notebook runs anywhere.

In [2]:
PATH = "/content/Telco-Customer-Churn.csv"

df = pd.read_csv(PATH)

In [3]:
df.shape

(7043, 21)

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
